# Retail Sales Analysis — Superstore Dataset

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

This notebook explores order-level retail sales data: sales, profit, discounts, products, customers, regions, and shipping.

**Before running:** save the provided dataset as `Superstore_Sales.csv` in the same folder as this notebook. The uploaded source was tab-separated text, so the loading cell supports both comma-separated CSV and tab-separated text.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')


## 2. Load the dataset

In [ ]:
file_path = Path("Superstore_Sales.csv")

if not file_path.exists():
    # Alternative filename if you keep the original uploaded text file beside this notebook
    alternatives = [Path("Pasted text(1).txt"), Path("Superstore_Sales.txt")]
    file_path = next((p for p in alternatives if p.exists()), file_path)

if not file_path.exists():
    raise FileNotFoundError(
        "Put your dataset beside this notebook and name it Superstore_Sales.csv "
        "(or keep the original Pasted text(1).txt filename)."
    )

# Detect delimiter from the first line
first_line = file_path.read_text(encoding="utf-8", errors="replace").splitlines()[0]
delimiter = "\\t" if "\\t" in first_line else ","
df = pd.read_csv(file_path, sep=delimiter)
df.head()


## 3. Initial inspection

In [ ]:
print("Shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
display(df.head())
display(df.info())
display(df.describe(include="all").T)


## 4. Data quality checks

In [ ]:
# Missing values by column
missing_summary = df.isna().sum().sort_values(ascending=False)
display(missing_summary[missing_summary > 0])

print("Duplicate full rows:", df.duplicated().sum())
print("Duplicate Row IDs:", df["Row ID"].duplicated().sum() if "Row ID" in df.columns else "Column not found")

# Keep Postal Code as a nullable string so leading zeros are not lost in future datasets.
if "Postal Code" in df.columns:
    df["Postal Code"] = df["Postal Code"].astype("Int64").astype("string")

# Parse dates using the format observed in the supplied data (day-month-year).
for col in ["Order Date", "Ship Date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format="%d-%m-%Y", errors="coerce")

for col in ["Sales", "Quantity", "Discount", "Profit"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Missing values after type conversion:")
display(df.isna().sum().sort_values(ascending=False).head(10))


## 5. Create analysis fields

In [ ]:
df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.to_period("M").astype("string")
df["Order Month Name"] = df["Order Date"].dt.month_name()

display(df[["Order Date", "Ship Date", "Shipping Days", "Sales", "Profit"]].head())


## 6. Overall KPIs

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_quantity = df["Quantity"].sum()
total_orders = df["Order ID"].nunique()
total_customers = df["Customer ID"].nunique()
profit_margin = (total_profit / total_sales * 100) if total_sales else np.nan

kpis = pd.DataFrame({
    "Metric": ["Total Sales", "Total Profit", "Total Quantity", "Unique Orders", "Unique Customers", "Profit Margin (%)"],
    "Value": [total_sales, total_profit, total_quantity, total_orders, total_customers, profit_margin]
})
display(kpis)


## 7. Sales and profit by category

In [ ]:
category_summary = (
    df.groupby("Category", dropna=False)
      .agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"),
           Units_Sold=("Quantity", "sum"), Orders=("Order ID", "nunique"))
      .sort_values("Total_Sales", ascending=False)
)
display(category_summary)
category_summary[["Total_Sales", "Total_Profit"]].plot(kind="bar", figsize=(9, 5), title="Sales and Profit by Category")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. Sub-category performance

In [ ]:
subcategory_summary = (
    df.groupby("Sub-Category", dropna=False)
      .agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"),
           Units_Sold=("Quantity", "sum"))
      .sort_values("Total_Profit")
)
display(subcategory_summary)
subcategory_summary["Total_Profit"].plot(kind="barh", figsize=(10, 8), title="Profit by Sub-Category")
plt.xlabel("Profit")
plt.tight_layout()
plt.show()


## 9. Monthly trend

In [ ]:
monthly_summary = (
    df.dropna(subset=["Order Date"])
      .groupby(df["Order Date"].dt.to_period("M"))
      .agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"),
           Orders=("Order ID", "nunique"))
      .reset_index(names="Month")
)
monthly_summary["Month"] = monthly_summary["Month"].dt.to_timestamp()
display(monthly_summary.head())
plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_summary, x="Month", y="Total_Sales", marker="o")
plt.title("Monthly Sales Trend")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 10. Regional performance

In [ ]:
region_summary = (
    df.groupby("Region", dropna=False)
      .agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"),
           Orders=("Order ID", "nunique"))
      .sort_values("Total_Sales", ascending=False)
)
display(region_summary)


## 11. Customer segment analysis

In [ ]:
segment_summary = (
    df.groupby("Segment", dropna=False)
      .agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"),
           Orders=("Order ID", "nunique"), Customers=("Customer ID", "nunique"))
      .sort_values("Total_Sales", ascending=False)
)
display(segment_summary)


## 12. Discount and profit

In [ ]:
discount_summary = (
    df.groupby("Discount", dropna=False)
      .agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"),
           Rows=("Row ID", "count"))
      .sort_index()
)
display(discount_summary)

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Discount", y="Profit", alpha=0.35)
plt.title("Discount vs Profit")
plt.tight_layout()
plt.show()


## 13. Best and worst products by profit

In [ ]:
product_summary = (
    df.groupby(["Product ID", "Product Name"], dropna=False)
      .agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"),
           Units_Sold=("Quantity", "sum"))
      .reset_index()
)
print("Top 10 products by profit")
display(product_summary.nlargest(10, "Total_Profit"))
print("Bottom 10 products by profit")
display(product_summary.nsmallest(10, "Total_Profit"))


## 14. Shipping analysis

In [ ]:
shipping_summary = (
    df.groupby("Ship Mode", dropna=False)
      .agg(Avg_Shipping_Days=("Shipping Days", "mean"),
           Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"),
           Orders=("Order ID", "nunique"))
      .sort_values("Orders", ascending=False)
)
display(shipping_summary)


## 15. Findings summary template

In [ ]:
# Use these outputs to write evidence-based findings after reviewing the results.
# Do not write a conclusion until you have checked the displayed tables and charts.
print("Analysis complete. Review the KPI, category, sub-category, monthly, regional, segment, discount, product, and shipping outputs above.")
